# Geospatial SDM in Python: April (Atlantic Sturgeon)

**Focus:** Species Distribution Modelling (SDM) with geospatial data, using April presence records for the Atlantic sturgeon (*Acipenser oxyrinchus*) and a stack of environmental rasters.

**Why this species?** Atlantic sturgeon are living fossils, a lineage that has changed little in around 120 million years, and they are also, right now, a conservation priority. 

Most of their Distinct Population Segments (DPS) along the US Atlantic coast have been listed as Endangered or Threatened under the Endangered Species Act since 2012, after over a century of decline driven largely by a 19th-century "caviar rush" that collapsed populations long before modern fisheries management existed. 

They are also slow to recover even once pressure is removed. Males typically do not mature until 5 to 24 years old, and females not until their teens to late twenties, so a single population can take decades to rebuild. That combination, a species we have already pushed to the edge and one that cannot simply bounce back, is exactly why knowing *where* suitable habitat exists actually matters. A **habitat suitability map** like the one you will build in this practical is not just a modelling exercise, it is the kind of evidence that feeds into critical habitat designations, dredging and construction permits, and offshore development reviews.

In this practical we build a full SDM workflow from the ground up. We load real presence points and a study extent, generate pseudo-absences, assemble an aligned stack of environmental predictor rasters, extract predictor values at each point to build a modelling table, train several classifier types, compare them honestly on held-out data, and combine them into an ensemble prediction that we map back across the study area. 

This is real **applied conservation science**. The maps you produce here are the kind that would feed into a habitat suitability assessment or an environmental impact report, so the choices we make at each step, how pseudo-absences are generated, which threshold we use to turn a probability into a decision, and how models are weighted in the ensemble, all have real consequences for the answer we hand to a manager or regulator.

*Note: this notebook expects the accompanying shapefiles and raster files (the `presence/`, `extent/` and `rasters/` subfolders) to be present locally alongside it. It is not set up to run in Google Colab, because the underlying raster data is too large to bundle into the course repository.*


## Learning Goals

By the end, you should be able to:
1. Load vector (`.shp`) and raster (`.tif`) geospatial data in Python.
2. Reproject data and create a spatial **buffer** to generate **pseudo-absences**.
3. Extract raster values at point locations to build a modelling table (`df`) with a **Presence** label.
4. Split data into training/validation/hold-out sets.
5. Train four classic classifiers (**Logistic Regression**, **Random Forest**, **SVM (RBF)** and **MLP**) for SDM.
6. Evaluate with ROC AUC and a confusion matrix for interpretation.
7. **Predict** across the raster stack and write **GeoTIFF** probability maps.
8. Build an **AUC-weighted ensemble** and evaluate it on a hold-out set.
9. Produce clear maps to visualise the predictions and presences/absences.



## Why this matters for Environmental Monitoring

- SDMs link **species observations** (presence/absence) with **environmental drivers** (depth, substrate, sea surface temperature, precipitation, photoperiod, and so on).
- Predictions support **habitat suitability** mapping, **conservation planning**, and **impact assessments**, all of which are decisions someone will act on.
- This notebook demonstrates a reproducible **geospatial machine learning** pipeline that is used widely in ecology, and that you will see again in almost any applied species or habitat modelling problem you encounter.


In [ ]:
# If running on a new environment, you may need to install a few packages.
# Remove the leading "!" and run if needed.
# !pip install geopandas shapely pyproj rasterio rioxarray xarray scikit-learn matplotlib numpy pandas scipy

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union
import rasterio
from rasterio.transform import Affine
import rioxarray as rxr
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

print("Imports OK")


In [ ]:
## Check working directory and change if necessary.
## Preferably keep it to the main working directory and adjust the paths as needed
os.getcwd()

## 0) Project Paths & Configuration

Before we touch any data, we set out the paths and settings the rest of the notebook depends on. Keeping these together at the top means that if the folder structure changes, or you want to rerun this for a different month, there is exactly one place to edit rather than variables scattered through fifty cells.


In [ ]:
# --- Keep original-like relative paths (as in your existing workflow) ---
PRESENCE_PATH = "presence/ats_apr.shp"
EXTENT_PATH   = "extent/ocs_extent.shp"

# Environmental rasters (April). Each one is a candidate predictor of sturgeon
# habitat suitability, and together they form the environmental stack we will
# extract values from and later predict across.
RASTER_PATHS = {
    "depth": "rasters/depth/depth.tif",                    # water depth
    "gravel": "rasters/substrate/Gravel_pct.tif",           # % gravel substrate, reference raster for alignment
    "mud": "rasters/substrate/Mud_pct.tif",                 # % mud substrate
    "photoperiod_apr": "rasters/photoperiod/apr_photoperiod.tif",  # day length in April
    "precip_apr": "rasters/precipitation/precip_apr.tif",   # April precipitation
    "sst_apr": "rasters/sst/apr_sst_ss.tif",                # April sea surface temperature
    "sss_apr": "rasters/sss_climatology_by_month/Apr/sss_climatology_Apr_resampled.tif",  # April sea surface salinity
}

# Reference raster we will align to. All other rasters get resampled onto
# this grid so that every layer shares the same resolution, extent and CRS,
# which is what lets us stack them and sample them at the same pixel.
REF_RASTER_PATH = RASTER_PATHS["gravel"]

In [ ]:

# Pseudo-absence settings
N_PSEUDO_ABS = 1000  # how many pseudo-absence points to generate
BUFFER_KM = 25        # "disk" buffer radius around each presence, in km
SEED = 42              # fixed random seed so buffering, sampling and model fitting are reproducible
np.random.seed(SEED)


In [ ]:

# Output filenames for the probability maps we will write later, one GeoTIFF per model plus the ensemble
OUT_RF  = "rf_apr.tif"
OUT_LR  = "lr_apr.tif"
OUT_SVM = "svm_apr.tif"
OUT_MLP = "mlp_apr.tif"
OUT_ENSEMBLE = "ensemble_apr.tif"


In [ ]:

print("Configured paths:")
for k, v in RASTER_PATHS.items():
    print(f"  {k} is found in {v}")


## 1) Load Presence Points (April) and Study Extent

We start with two vector layers: the sturgeon **presence points** recorded in April, and the **study extent** polygon that defines the area we are allowed to model within. Everything downstream, pseudo-absence sampling, raster extraction, prediction, is bounded by this extent, so it is worth looking at both layers before doing anything else.


In [ ]:

# Load presence points (April) and study extent
presences = gpd.read_file(PRESENCE_PATH)
extent = gpd.read_file(EXTENT_PATH)

print(f"Presences: {len(presences)} points")
display(presences.head)
display(extent.head)


In [ ]:

# Ensure we have a CRS (Coordinate Reference System). Without one, distances,
# areas and buffers computed later would be meaningless, since we would not
# know what units the coordinates are actually in.
if presences.crs is None:
    raise ValueError("Presence shapefile lacks a CRS. Please define it before proceeding.")
if extent.crs is None:
    raise ValueError("Extent shapefile lacks a CRS. Please define it before proceeding.")


In [ ]:

# For mapping + overlay, work in WGS84 (EPSG:4326) for easy viewing,
# but we will temporarily go to a meter-based CRS when buffering.
presences = presences.to_crs(epsg=4326)
extent = extent.to_crs(epsg=4326)


In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
extent.boundary.plot(ax=ax, linewidth=1)
presences.plot(ax=ax, markersize=8, color = "red")
ax.set_title("April presences within study extent")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()



## 2) Create a 25 km Buffer Around Presences and Generate Pseudo-Absences

An SDM needs both presences and absences to learn a decision boundary, but true confirmed absences are rare in survey data (a species could simply have been missed rather than genuinely absent). The standard workaround is a **pseudo-absence**: a location we treat as "not observed present" because it is far enough from any known presence that the risk of it secretly being suitable habitat is low.

Why buffer?
- We avoid sampling pseudo-absences **too close** to known presences, because habitat right next to a confirmed sighting is exactly where the species' true range boundary is uncertain, and calling that a confident absence would mislead the model.
- We use a **25 km disk** around each presence as an exclusion zone, then **sample randomly** in the remaining area inside the study extent. This gives the models a genuine contrast between "known suitable" and "plausibly unsuitable" locations to learn from.


In [ ]:

# Choose a meter-based CRS for accurate buffering, since buffering in degrees
# (WGS84) would distort distances differently in the x and y direction.
# Web Mercator (EPSG:3857) is OK for ~25 km buffers at mid-latitudes, though for production use you would prefer a local UTM zone.
presences_m = presences.to_crs(epsg=3857)
extent_m = extent.to_crs(epsg=3857)
display(presences_m)
display(extent_m)


In [ ]:

# Create a 25 km buffer around all presence points and union them into a
# single exclusion polygon (overlapping buffers around nearby presences
# should not be treated as separate shapes)
buffer_m = presences_m.buffer(BUFFER_KM * 1000.0)
buffer_union_m = gpd.GeoSeries([unary_union(buffer_m.geometry)], crs=extent_m.crs)
display(buffer_m)
display(buffer_union_m)


In [ ]:

# Area where absences are allowed: extent minus the buffer exclusion zone
extent_polygon_m = unary_union(extent_m.geometry)
absence_area_m = extent_polygon_m.difference(buffer_union_m.iloc[0])

# Convert absence area back to WGS84 for random sampling convenience
absence_area = gpd.GeoSeries([absence_area_m], crs=extent_m.crs).to_crs(epsg=4326).iloc[0]
absence_area


In [ ]:

# Sample random points inside a polygon by rejection sampling on its bounding box.
# This is simple and robust for irregular polygons: draw a random point in the
# box, keep it only if it actually falls inside the polygon, and repeat.
def random_points_in_polygon(polygon, n, seed=SEED, max_iter=200000):
    rng = np.random.default_rng(seed)  # random number generator (reproducible)
    minx, miny, maxx, maxy = polygon.bounds  # axis-aligned bounding box of the polygon: (minx, miny, maxx, maxy)
    points = []
    count = 0  # how many candidate draws we've tried
    while len(points) < n and count < max_iter:  # draw a candidate point uniformly in the bbox
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(x, y)
        if polygon.contains(p):         # accept the point only if it is strictly inside the polygon
            points.append(p)
        count += 1
    if len(points) < n:  # warn if we did not manage to get enough points
        print(f"Warning: Only generated {len(points)} of {n} requested points. Consider increasing max_iter.")
    return points


In [ ]:
# Generate pseudo-absence points inside the allowed area (extent minus the 25 km buffer)
absence_points = random_points_in_polygon(absence_area, N_PSEUDO_ABS, seed=SEED)
display(absence_points)

In [ ]:

# Give an index so the scalar broadcasts, and label every pseudo-absence with Presence=0
n = len(absence_points)
absences = gpd.GeoDataFrame(
    {"Presence": 0},
    index=range(n),
    geometry=absence_points,
    crs="EPSG:4326",
)


In [ ]:
display(absences.head())
display(presences.head())

In [ ]:

# Tag presences with Presence=1 to match modelling table later
presences_tagged = presences.copy()
presences_tagged["Presence"] = 1
display(presences_tagged)


In [ ]:

# Reproject the buffer back to WGS84 purely so we can plot it on the same map as everything else
buffer_4326 = gpd.GeoDataFrame(geometry=[buffer_union_m.to_crs(epsg=4326).iloc[0]], crs="EPSG:4326")
buffer_4326


In [ ]:

fig, ax = plt.subplots(figsize=(6, 10))
extent.boundary.plot(ax=ax, linewidth=1, label="Extent")

buffer_4326.boundary.plot(ax=ax, color="black", linewidth=2, label="25 km buffer")

presences_tagged.plot(ax=ax, markersize=8, label="Presence", color="green")
absences.plot(ax=ax, markersize=4, label="Absence (sample)", color="red")

ax.set_title("Buffer and pseudo-absences")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.show()


In [ ]:

# Ensure WGS84 for correct Lon/Lat
absences = absences.to_crs(epsg=4326).copy()

# Add coordinate columns from geometry
absences["Longitude"] = absences.geometry.x
absences["Latitude"]  = absences.geometry.y

# Enforce int dtype and reorder columns to match presences, so the two
# tables can be concatenated cleanly later
absences["Presence"] = absences["Presence"].astype(int)
absences = absences[["Latitude", "Longitude", "Presence", "geometry"]]
display(absences)


In [ ]:

# (If needed, do the same for presences_tagged to guarantee consistency.
# Left commented out here because presences_tagged already carries the
# columns we need in the right order once we extract raster values below.)
# presences_tagged = presences_tagged.to_crs(epsg=4326).copy()
# presences_tagged["Longitude"] = presences_tagged.geometry.x
# presences_tagged["Latitude"]  = presences_tagged.geometry.y
# presences_tagged = presences_tagged[["Latitude", "Longitude", "Presence", "geometry"]]


## 3) Read and Align Environmental Rasters into `layers`

Each environmental raster was produced separately (depth from bathymetry, substrate from sediment surveys, SST from satellite composites, and so on), so they are very unlikely to share the same resolution, extent or even coordinate reference system. Before we can stack them into a single predictor array, or sample them at the same point, every raster has to sit on **exactly the same grid**. This section reprojects and resamples every layer onto the reference raster's grid (the gravel substrate raster, chosen arbitrarily as the alignment target) so that pixel `(i, j)` means the same location in every layer.


In [ ]:

# We'll align all rasters to the reference raster (gravel) using rioxarray
# and store an ordered dictionary-like mapping in `layers`.

from collections import OrderedDict

# Open the reference raster once. masked=True turns the raster's nodata
# value into proper NaNs, and squeeze() drops the singleton band dimension
# for a single-band raster.
ref = rxr.open_rasterio(REF_RASTER_PATH, masked=True).squeeze()  # ref raster path is the GRAVEL raster
if ref.rio.crs is None:
    ref = ref.rio.write_crs("EPSG:4326")  # or the correct CRS for your data


In [ ]:
ref

In [ ]:
from rasterio.enums import Resampling

# How each layer should be resampled onto the reference grid. Bilinear
# interpolation is appropriate for continuous variables like depth, SST,
# precipitation and percentage substrate cover, since it produces a smooth
# blend of neighbouring pixel values. A categorical layer (for example a
# discrete land-cover class) should instead use nearest-neighbour resampling,
# so as not to invent fractional "in-between" classes that do not exist.
resampling_map = {
    "depth": Resampling.bilinear,
    "gravel": Resampling.bilinear,
    "mud": Resampling.bilinear,
    "photoperiod_apr": Resampling.bilinear,
    "precip_apr": Resampling.bilinear,
    "sst_apr": Resampling.bilinear,
    "sss_apr": Resampling.bilinear,
    # "landcover": Resampling.nearest,  # example for categorical
}

In [ ]:

# This is the key alignment step: for every raster in RASTER_PATHS, open it,
# make sure it has a CRS, then reproject_match it onto the reference grid
# (ref) using the chosen resampling method. After this loop, every array in
# `layers` shares the same shape, resolution, extent and CRS, so they can be
# safely stacked band-by-band and indexed with the same (row, col).
layers = OrderedDict()
for name, path in RASTER_PATHS.items():
    da = rxr.open_rasterio(path, masked=True).squeeze()
    if da.rio.crs is None:
        da = da.rio.write_crs(ref.rio.crs)
    resampler = resampling_map.get(name, Resampling.bilinear)  # <-- enum object
    da_matched = da.rio.reproject_match(ref, resampling=resampler)
    layers[name] = da_matched.astype("float32").rio.write_nodata(np.nan)


In [ ]:
layers

In [ ]:

# Plot each aligned layer on its own so we can sanity check that the
# reprojection looks sensible (correct extent, no obvious artefacts)
for i, (nm, da) in enumerate(list(layers.items()), start=1):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(da.values, extent=[da.x.min(), da.x.max(), da.y.min(), da.y.max()], origin="upper")
    ax.set_title(f"{nm}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, shrink=0.7)
    plt.show()


In [ ]:
presences_tagged, absences

In [ ]:

## Fully stack our rasters
# After you've built `layers`, stack them along a new leading axis so we get
# one array of shape (bands, rows, cols), the standard layout for a raster stack.
layer_names = list(layers.keys())                 # keep a fixed band order
stack_np = np.stack([layers[n].values for n in layer_names], axis=0)  # shape: (B, Y, X)

# Mask: True where ANY band is NaN at that pixel. A pixel is only usable for
# prediction later if every predictor has a real value there.
mask = np.any(np.isnan(stack_np), axis=0)         # shape: (Y, X)

# Coordinates (from any layer, since they are all aligned to the same grid)
x = layers[layer_names[0]].x.values
y = layers[layer_names[0]].y.values
extent = [x.min(), x.max(), y.min(), y.max()]


In [ ]:
layer_names

In [ ]:

### Plot the stack
import numpy as np
import matplotlib.pyplot as plt
import math

# Plot each band as a small multiple, so we can compare all predictors side by side
B = stack_np.shape[0]
cols = min(3, B)                     # wrap to 3 columns
rows = math.ceil(B / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows), squeeze=False)

for i in range(B):
    r, c = divmod(i, cols)
    ax = axes[r][c]
    arr = np.ma.masked_invalid(stack_np[i])  # hide NaNs (nodata)
    im = ax.imshow(arr, extent=extent, origin="upper")
    ax.set_title(layer_names[i])
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, shrink=0.75)

# Hide any empty axes if B isn't a multiple of cols
for j in range(B, rows*cols):
    r, c = divmod(j, cols)
    axes[r][c].axis("off")

fig.suptitle("Environmental Raster Stack (bands)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:

# If needed: pip install ipywidgets, then enable the Jupyter extension if required
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt
import numpy as np

# A small interactive viewer so you can flip through bands one at a time,
# which is often easier to inspect closely than the small-multiples grid above
def show_band(i=0):
    fig, ax = plt.subplots(figsize=(6,5))
    arr = np.ma.masked_invalid(stack_np[i])
    im = ax.imshow(arr, extent=extent, origin="upper")
    ax.set_title(f"Band {i}: {layer_names[i]}")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Value")
    plt.show()


In [ ]:
interact(show_band, i=IntSlider(min=0, max=len(layer_names)-1, step=1, value=0))

In [ ]:

## Save the raster stack
# Writing the aligned stack to disk as a single multi-band GeoTIFF means we
# do not have to repeat the alignment step every time we reopen this
# notebook, we can just load env_stack.tif directly.
import rasterio

with rasterio.open(REF_RASTER_PATH) as src_ref:
    profile = src_ref.profile
profile.update(count=len(layer_names), dtype="float32", nodata=np.nan, compress="lzw")

with rasterio.open("env_stack.tif", "w", **profile) as dst:
    for i, nm in enumerate(layer_names, start=1):
        dst.write(layers[nm].values.astype("float32"), i)
        dst.set_band_description(i, nm)


## Check if you can load the environmental raster stack

A quick check that the file we just wrote reopens correctly, with the right number of bands, band names and metadata, before we build anything else on top of it.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
import rasterio

STACK_PATH = "env_stack.tif"   # change if needed

# Open the stack as (band, y, x)
stack_xr = rxr.open_rasterio(STACK_PATH, masked=True).squeeze()
stack_xr

In [ ]:

# Get human-readable band names (descriptions) from the file, if present,
# and use them as coordinate labels so we can select bands by name later
with rasterio.open(STACK_PATH) as src:
    band_desc = list(src.descriptions)
band_desc = [bd if bd not in (None, "") else f"band_{i+1}" for i, bd in enumerate(band_desc)]
stack_xr = stack_xr.assign_coords(band=band_desc)


In [ ]:

# Plot each band as its own subplot
B = stack_xr.sizes["band"]
cols = min(3, B)                          # up to 3 columns, adjust if you like
rows = math.ceil(B / cols)


In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows), squeeze=False)

for i, name in enumerate(band_desc):
    r, c = divmod(i, cols)
    da = stack_xr.sel(band=name)
    img = np.ma.masked_invalid(da.values.astype("float32"))
    extent = [float(da.x.min()), float(da.x.max()), float(da.y.min()), float(da.y.max())]

    im = axes[r][c].imshow(img, extent=extent, origin="upper")
    axes[r][c].set_title(str(name))
    axes[r][c].set_xlabel("Longitude")
    axes[r][c].set_ylabel("Latitude")
    plt.colorbar(im, ax=axes[r][c], shrink=0.75)

fig.suptitle("Raster Stack: Bands", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:

# Per-band descriptions (names), if they were written, plus the raster's
# core georeferencing metadata. Worth checking this once: if the CRS,
# resolution or bounds look wrong here, everything downstream will be wrong too.
print("\n=== Band descriptions ===")
print(src.descriptions)                 # tuple of strings per band
print("\n")
print("=== Basic info ===")
print("CRS:        ", src.crs)          # coordinate reference system
print("Width x Height:", src.width, "x", src.height)
print("Bands (count):", src.count)
print("Dtype:       ", src.dtypes[0])
print("NoData:      ", src.nodata)
print("Resolution:  ", src.res)         # (pixel_size_x, pixel_size_y)
print("Bounds:      ", src.bounds)      # (minx, miny, maxx, maxy)
print("Transform:   ", src.transform)   # affine geotransform


## 4) Extract Raster Values at Points and Build the Modelling Table `df`

This is the step where geospatial data becomes a plain modelling table. For every presence and pseudo-absence point, we look up the environmental predictor values (depth, substrate, SST, and so on) of the raster pixel that point falls in, and write them into a row alongside the `Presence` label. The result, `df_w_var`, is exactly the kind of feature table a classifier expects: one row per observation, one column per predictor, one target column. Everything geospatial about the problem is now encoded in those predictor values, the models themselves never see a coordinate or a raster again.


In [ ]:

# Check layers are aligned to the same grid/CRS before we trust point
# extraction to use a single shared transform
ref_da = next(iter(layers.values()))
ref_crs = ref_da.rio.crs
ref_shape = ref_da.shape

print(ref_da)
print(ref_crs)
print(ref_shape)


In [ ]:
layers.keys()

In [ ]:

for nm, da in layers.items():
    # If anything mismatches, force it to match the reference (belt and braces
    # check, since reproject_match earlier should already have handled this)
    if (str(da.rio.crs) != str(ref_crs)) or (da.shape != ref_shape):
        layers[nm] = da.rio.reproject_match(ref_da)


In [ ]:
absences

In [ ]:

# Concatenate presence + absence points into one table of observations,
# which is what we will loop over to extract predictor values
all_points = pd.concat([presences_tagged, absences], ignore_index=True)
display(all_points)


In [ ]:

# Define a point extractor
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.transform import rowcol

def extract_layers_to_points_simple(points_gdf, layers_dict, add_xy=True):
    """
    Extract each raster layer's value at each point.
    Works on the reprojected, aligned DataArrays in `layers_dict`.
    """
    # Reproject points to the raster grid CRS (shared by all layers), since
    # rowcol() below needs point coordinates in the same CRS as the raster
    ref = next(iter(layers_dict.values()))
    pts = points_gdf.to_crs(ref.rio.crs).reset_index(drop=True)

    # Prepare output table
    out = pd.DataFrame(index=np.arange(len(pts)))
    if "Presence" in pts.columns:
        out["Presence"] = pts["Presence"].astype(int).values

    # Keep human-friendly Lon/Lat alongside, for later plotting and QA
    if add_xy:
        pts_ll = pts.to_crs(epsg=4326)
        out["Latitude"]  = pts_ll.geometry.y.values
        out["Longitude"] = pts_ll.geometry.x.values

    # Coordinates to sample, in the raster's own CRS
    xs = pts.geometry.x.values
    ys = pts.geometry.y.values

    # For each layer, convert (x, y) to (row, col) using its affine
    # transform, then sample the array value at that pixel
    for lname, da in layers_dict.items():
        t = da.rio.transform()
        h, w = da.shape

        # vectorised map from coordinates to raster indices
        rows, cols = rowcol(t, xs, ys)

        # mask out-of-bounds indices safely (a point could fall just outside
        # the raster extent due to reprojection rounding)
        valid = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
        vals = np.full(len(pts), np.nan, dtype=float)
        # rioxarray opened with masked=True, so nodata is already NaN inside da.values
        vals[valid] = da.values[rows[valid], cols[valid]]

        out[lname] = vals

    return out


In [ ]:

## Extract predictor values from our raster layers at every presence and pseudo-absence point
df_w_var = extract_layers_to_points_simple(all_points, layers, add_xy=True)
display(df_w_var)


In [ ]:

# Quick QA: how many rows have a missing value in at least one column?
# Points that fall on a nodata pixel (coastline edge effects, small
# misalignments) will show up here.
print(df_w_var.head())
print(df_w_var.isna().sum())   # typically we drop rows with any NaNs


In [ ]:

df_w_var = df_w_var.dropna().reset_index(drop=True)

### We need to drop Latitude and Longitude before modelling. If we left them
### in as predictors, the models could effectively memorise "where the known
### presences are" rather than learning the environmental relationship, which
### would look like excellent performance here but would fail to generalise
### to any location we have not already surveyed.
df_w_var = df_w_var.drop(columns=['Latitude', 'Longitude'])
display(df_w_var)


## 5) Train/Test/Hold-out Split

We split the data three ways rather than the usual two, because this notebook uses the test set for two different jobs: comparing models against each other, and (later) weighting them in the ensemble. If we then evaluated the ensemble on that same test set, the result would be optimistic, since the weights were chosen with knowledge of exactly that data. The **hold-out** set is never touched until the very end, so it gives us one honest, unbiased number for how the final ensemble is likely to perform on new locations.


In [ ]:

# We'll create a 10% hold-out set first, then split the remaining into train/test (80/20).
# Keep variable names used in your original workflow: X_train, X_test, y_train, y_test, X_hold, y_hold.

features = [c for c in df_w_var.columns if c not in ["Presence"]]
X = df_w_var[features].values
y = df_w_var["Presence"].values

# 10% hold-out, stratified so presence/absence proportions are preserved in each split
X_rest, X_hold, y_rest, y_hold = train_test_split(X, y, test_size=0.10, random_state=SEED, stratify=y)

# of the remaining, 80/20 train/test
X_train, X_test, y_train, y_test = train_test_split(X_rest, y_rest, test_size=0.20, random_state=SEED, stratify=y_rest)

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Hold-out: {X_hold.shape}")


In [ ]:
print(f"DataFrame for X:\n {pd.DataFrame(X).head()}")

In [ ]:
print(f"DataFrame for X:\n {pd.DataFrame(X_train).head()}")
print("\n")
print(f"DataFrame for y (Target):\n {pd.DataFrame(y_train).head()}")

## 6) Train Four Models: Logistic Regression, Random Forest, SVM (RBF), MLP

We deliberately train four quite different classifiers rather than committing to one, because each brings a different assumption about how the environmental predictors relate to presence, and in practice we rarely know in advance which assumption fits the data best.

- **Logistic Regression** fits a smooth, linear decision boundary in the (scaled) predictor space. It is the simplest model here, and its coefficients are directly interpretable (which predictors push probability up or down), which makes it a useful, honest baseline to beat.
- **Random Forest** is an ensemble of decision trees, each trained on a bootstrapped sample with a random subset of features considered at each split. It naturally captures non-linear relationships and interactions between predictors (for example, "shallow water is only suitable if the substrate is gravel") without us having to specify them, and it needs no feature scaling.
- **SVM (RBF)** uses a radial basis function kernel to project the predictors into a higher-dimensional space where a flexible, curved decision boundary becomes possible. It can capture complex shapes in predictor space, but is sensitive to feature scaling and to the choice of kernel hyperparameters.
- **MLP** (a small feed-forward neural network) is the most flexible model of the four, able to approximate very complex decision surfaces, but it also has the most capacity to overfit on a dataset this size, which is why we add regularisation and early stopping below.

Comparing these on the same train/test split later tells us whether the extra flexibility of SVM or MLP is actually earning its keep here, or whether the simpler models do just as well. It is tempting to treat this as a leaderboard exercise, chase the highest AUC and stop there, but for a threatened species that temptation is worth resisting: a model that is 1% more accurate but tells you nothing about *why* it predicts what it predicts is much less useful to a manager than a slightly less accurate one you can actually explain. That is exactly the gap the Explainable AI section later in this notebook is there to close.

In [ ]:

# We'll wrap scale-sensitive models (LR, SVM) with a StandardScaler in a Pipeline,
# so that every predictor contributes on a comparable scale and the optimiser
# is not dominated by whichever variable happens to have the largest raw units.
# RandomForest does not require scaling, since tree splits only care about
# the ordering of values within a feature, not their absolute scale.

models = {}

models["lr"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED))
])


In [ ]:

models["rf"] = RandomForestClassifier(
    n_estimators=300,    # number of trees in the forest, more trees generally means a more stable average prediction
    max_depth=None,      # let trees grow until leaves are pure or too small to split further
    random_state=SEED,
    n_jobs=-1             # use all available CPU cores to fit trees in parallel
)


In [ ]:

models["svm_rbf"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", probability=True, random_state=SEED))
    # probability=True is required so predict_proba is available below, but it
    # adds computational cost (an internal cross-validated calibration step)
])


In [ ]:

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42  # keep your existing seed

models["mlp"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(32, 32),  # two hidden layers of 32 units each
        activation="relu",
        solver="adam",
        alpha=1e-4,                   # L2 regularisation, penalises large weights to reduce overfitting
        max_iter=200,                 # increase if it hasn't converged
        random_state=SEED,
        early_stopping=True,          # stop training once validation performance plateaus, a simple generalisation safeguard
        n_iter_no_change=10,
        validation_fraction=0.1,
    ))
])


In [ ]:
models

### Evaluate each model in `models` on the test set using ROC AUC

We use ROC AUC (area under the receiver operating characteristic curve) rather than plain accuracy to compare models, because it summarises how well a model ranks presences above absences across every possible threshold, not just at one arbitrary cutoff. This matters here because we have not yet decided what threshold we will use to turn a probability into a binary prediction, that choice comes later.

This assumes:
- `models` is a dict like `{"lr": Pipeline(...), "rf": RandomForestClassifier(...), "svm_rbf": Pipeline(...), ...}`
- `X_train`, `y_train`, `X_test`, `y_test` are already defined

For each model we:
- fit it on the training data
- produce a probability-like score for class 1 (Presence=1)
- compute ROC AUC and store the result for comparison


In [ ]:
# Results containers (fresh each time you run this block)
aucs = {}      # holds AUC per model name
y_probas = {}  # optional: keep probabilities for plots/ensembles

In [ ]:

# LOGISTIC REGRESSION
# Assume models["lr"] is a Pipeline(StandardScaler for LogisticRegression)

mdl_lr = models["lr"]              # pick the model
mdl_lr.fit(X_train, y_train)       # 1) fit

# Get probability for class 1 (Presence=1)
y_proba_lr = mdl_lr.predict_proba(X_test)[:, 1]

# 3) evaluate with ROC AUC
auc_lr = roc_auc_score(y_test, y_proba_lr)

# 4) store + print
aucs["lr"] = auc_lr
y_probas["lr"] = y_proba_lr
print(f"lr: Test AUC = {auc_lr:.3f}")


In [ ]:

# RANDOM FOREST
# Assumes models["rf"] is a RandomForestClassifier

mdl_rf = models["rf"]
mdl_rf.fit(X_train, y_train)

y_proba_rf = mdl_rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_proba_rf)

aucs["rf"] = auc_rf
y_probas["rf"] = y_proba_rf
print(f"rf: Test AUC = {auc_rf:.3f}")


In [ ]:

# SVM (RBF)
# Assumes models["svm_rbf"] is a Pipeline(StandardScaler -> SVC(kernel="rbf", probability=True))

mdl_svm = models["svm_rbf"]

# Ensure we have probabilities available
svm_has_proba = hasattr(getattr(mdl_svm, "predict_proba", None), "__call__")

mdl_svm.fit(X_train, y_train)

y_proba_svm = mdl_svm.predict_proba(X_test)[:, 1]
auc_svm = roc_auc_score(y_test, y_proba_svm)

aucs["svm_rbf"] = auc_svm
y_probas["svm_rbf"] = y_proba_svm
print(f"svm_rbf: Test AUC = {auc_svm:.3f}")


In [ ]:

## MLP
# Fit & evaluate like your other models
mdl_mlp = models["mlp"]
mdl_mlp.fit(X_train, y_train)
y_proba_mlp = mdl_mlp.predict_proba(X_test)[:, 1]
auc_mlp = roc_auc_score(y_test, y_proba_mlp)
aucs["mlp"] = auc_mlp
print(f"mlp: AUC = {auc_mlp:.3f}")


In [ ]:

print("AUCs:", aucs)

# Identify best by AUC (useful for picking which confusion matrix to show next,
# and this ranking is also what the ensemble weighting will be based on later)
best_name = max(aucs, key=aucs.get)
best_auc = aucs[best_name]
print(f"Best on test set: {best_name} (AUC = {best_auc:.3f})")


In [ ]:
# A quick visual comparison of the four models' test-set AUCs, easier to scan at a
# glance than the printed dict above, and a natural place to spot whether one model
# is clearly out ahead or whether they are all bunched close together
fig, ax = plt.subplots(figsize=(6, 4))
auc_series = pd.Series(aucs).sort_values(ascending=False)
bars = ax.bar(auc_series.index, auc_series.values, color="tab:blue")
bars[0].set_color("tab:orange")  # highlight the best-performing model

ax.set_ylim(0, 1)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, label="Random guessing (AUC = 0.5)")
ax.set_ylabel("Test AUC")
ax.set_title("Model Comparison: Test-Set AUC")
for x, v in zip(auc_series.index, auc_series.values):
    ax.text(x, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Classification Report for each Model

ROC AUC tells us how well each model ranks presences above absences overall, but it does not tell us what happens once we actually commit to a decision threshold. The confusion matrices below make that concrete: at a chosen cutoff, how many presences do we correctly flag, and how many false alarms do we generate?


In [ ]:

# Helper to compute metrics from a confusion matrix
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score

def metrics_from_cm(cm: np.ndarray):
    """
    Given a 2x2 confusion matrix with layout:
        [[TN, FP],
         [FN, TP]]
    return: Accuracy, Sensitivity (TPR), Specificity (TNR), PPV, NPV

    Ecologically, sensitivity (how many true presences we correctly flag) and
    specificity (how many true absences we correctly flag) are the pair to
    watch: a model can look accurate overall while still missing most real
    presences, if presences are the rarer class.
    """
    tn, fp, fn, tp = cm.ravel()
    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total if total > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan     # aka recall/TPR
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan     # aka TNR
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan             # precision
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    return accuracy, sensitivity, specificity, ppv, npv


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score

def plot_cm(cm, class_names=('Absence','Presence'), title='Confusion matrix', normalize=False):
    """
    Plot a 2x2 confusion matrix.
    cm layout expected: [[TN, FP],
                         [FN, TP]]
    normalize=True shows per-row (true-class) proportions.
    """
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_disp = cm.astype(float) / np.maximum(row_sums, 1)
    else:
        cm_disp = cm

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_disp, interpolation='nearest', cmap='Set2')  # default colormap
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names, yticklabels=class_names,
        xlabel='Predicted label', ylabel='True label',
        title=title
    )
    # Show numbers in each cell
    fmt = '.2f' if normalize else 'd'
    thresh = cm_disp.max() / 2.0 if cm_disp.size else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm_disp[i, j], fmt),
                ha='center', va='center',
                color='white' if cm_disp[i, j] > thresh else 'black'
            )
    plt.tight_layout()
    plt.show()


### Evaluate each model with a sensible threshold

A model's `predict_proba` gives a continuous suitability score between 0 and 1, but a habitat map or a yes/no conservation decision needs a single cutoff. Here we use the conventional default of **0.5**: any pixel or point scoring 0.5 or above is called "presence". This is a reasonable starting point when we have no strong reason to weight false positives and false negatives differently, but it is still an assumption, not a fact, and a real management application might deliberately choose a lower threshold to avoid missing rare presences, accepting more false alarms in exchange.

- If a model has `predict_proba`, we use 0.5 as the probability threshold.
- If not (for example, an SVM without `probability=True`), we would use the decision function and threshold at 0 (the margin) instead.


In [ ]:

## Logistic Regression
# Assumes: models["lr"] exists and is already fitted OR fit now
mdl_lr = models["lr"]
mdl_lr.fit(X_train, y_train)  # safe to call, refits if needed

# Probability of class 1 (Presence)
y_proba_lr = mdl_lr.predict_proba(X_test)[:, 1]

# Threshold at 0.5 for a simple confusion matrix
y_pred_lr = (y_proba_lr >= 0.5).astype(int)

# Confusion matrix and metrics
cm_lr = confusion_matrix(y_test, y_pred_lr)   # [[TN, FP], [FN, TP]]
acc_lr, sens_lr, spec_lr, ppv_lr, npv_lr = metrics_from_cm(cm_lr)

# AUC uses the *scores* (probabilities here), not the thresholded labels
auc_lr = roc_auc_score(y_test, y_proba_lr)

print("Logistic Regression: Confusion matrix [[TN, FP],[FN, TP]]:\n", cm_lr, "\n")
plot_cm(cm_lr, class_names=('Absence','Presence'), title='Logistic Regression: Confusion Matrix', normalize=False)
print(f"Accuracy={acc_lr:.3f} | Sensitivity={sens_lr:.3f} | Specificity={spec_lr:.3f} | "
      f"PPV={ppv_lr:.3f} | NPV={npv_lr:.3f} | AUC={auc_lr:.3f}")


In [ ]:

## Random Forests
mdl_rf = models["rf"]
mdl_rf.fit(X_train, y_train)

y_proba_rf = mdl_rf.predict_proba(X_test)[:, 1]
y_pred_rf  = (y_proba_rf >= 0.5).astype(int)

cm_rf = confusion_matrix(y_test, y_pred_rf)
acc_rf, sens_rf, spec_rf, ppv_rf, npv_rf = metrics_from_cm(cm_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print("Random Forest: Confusion matrix [[TN, FP],[FN, TP]]:\n", cm_rf, "\n")
plot_cm(cm_rf, class_names=('Absence','Presence'), title='Random Forests: Confusion Matrix', normalize=False)
print(f"Accuracy={acc_rf:.3f} | Sensitivity={sens_rf:.3f} | Specificity={spec_rf:.3f} | "
      f"PPV={ppv_rf:.3f} | NPV={npv_rf:.3f} | AUC={auc_rf:.3f}")


In [ ]:

## SVM
mdl_svm = models["svm_rbf"]
mdl_svm.fit(X_train, y_train)

# If your SVM was defined with probability=True, use probabilities:
y_proba_svm = mdl_svm.predict_proba(X_test)[:, 1]
y_pred_svm  = (y_proba_svm >= 0.5).astype(int)

cm_svm = confusion_matrix(y_test, y_pred_svm)
acc_svm, sens_svm, spec_svm, ppv_svm, npv_svm = metrics_from_cm(cm_svm)
auc_svm = roc_auc_score(y_test, y_proba_svm)

print("SVM (RBF): Confusion matrix [[TN, FP],[FN, TP]]:\n", cm_svm, "\n")
plot_cm(cm_svm, class_names=('Absence','Presence'), title='Support Vector Machines: Confusion Matrix', normalize=False)
print(f"Accuracy={acc_svm:.3f} | Sensitivity={sens_svm:.3f} | Specificity={spec_svm:.3f} | "
      f"PPV={ppv_svm:.3f} | NPV={npv_svm:.3f} | AUC={auc_svm:.3f}")


In [ ]:

## MULTILAYER PERCEPTRON (MLP)
# Assumes: models["lr"] exists and is already fitted OR fit now
mdl_mlp = models["mlp"]
mdl_mlp.fit(X_train, y_train)  # safe to call, refits if needed

# Probability of class 1 (Presence)
y_proba_mlp = mdl_mlp.predict_proba(X_test)[:, 1]

# Threshold at 0.5 for a simple confusion matrix
y_pred_mlp = (y_proba_mlp >= 0.5).astype(int)

# Confusion matrix and metrics
cm_mlp = confusion_matrix(y_test, y_pred_mlp)   # [[TN, FP], [FN, TP]]
acc_mlp, sens_mlp, spec_mlp, ppv_mlp, npv_mlp = metrics_from_cm(cm_mlp)

# AUC uses the *scores* (probabilities here), not the thresholded labels
auc_mlp = roc_auc_score(y_test, y_proba_mlp)

print("Multilayer Perceptron: Confusion matrix [[TN, FP],[FN, TP]]:\n", cm_mlp, "\n")
plot_cm(cm_mlp, class_names=('Absence','Presence'), title='Multilayer Perceptron: Confusion Matrix', normalize=False)
print(f"Accuracy={acc_mlp:.3f} | Sensitivity={sens_mlp:.3f} | Specificity={spec_mlp:.3f} | "
      f"PPV={ppv_mlp:.3f} | NPV={npv_mlp:.3f} | AUC={auc_mlp:.3f}")


## 7) Predict Over the Raster Stack and Write GeoTIFFs

So far every model has only ever seen the predictor values at our sample points. The real value of an SDM is turning that into a **continuous prediction across the whole study area**: for every pixel in the raster stack, feed its predictor values through the fitted model and get back a probability of presence. Stitching those pixel-by-pixel probabilities back into a raster gives us a habitat suitability map, the actual deliverable of this kind of analysis.

To predict over rasters we:
- stack the matched raster layers into an array of shape (pixels, bands),
- apply each fitted model to every valid pixel (carefully excluding pixels where any predictor is nodata),
- and write the resulting probability surface out as a GeoTIFF using the reference raster's georeferencing metadata, so it lines up correctly in a GIS.


In [ ]:

# Build 3D stack [bands, y, x] -> then reshape to [n_pixels, n_bands], which
# is the row-per-observation layout every scikit-learn model expects
layer_names = list(layers.keys())  # use this order consistently, it must match the order features were trained on
stack = np.stack([layers[nm].values for nm in layer_names], axis=0)  # [B, Y, X]
mask = np.any([np.isnan(arr) for arr in stack], axis=0)              # True where any band is NaN

B, Y, X = stack.shape
flat = stack.reshape(B, -1).T  # [N, B], one row per pixel
valid_idx = ~mask.ravel()      # only pixels where every predictor has a value are valid for prediction


In [ ]:

# Prediction helper that uses true probabilities. We only predict for the
# valid pixels (valid_idx) and leave everything else as NaN, so nodata areas
# (open ocean beyond the study extent, missing substrate data, etc.) stay
# transparent in the output map rather than getting a fabricated value.
def predict_map_proba(model):
    """
    Predict P(class=1) for each valid pixel.
    Requires the model to implement predict_proba.
    """
    out = np.full(flat.shape[0], np.nan, dtype="float32")
    proba = model.predict_proba(flat[valid_idx])[:, 1].astype("float32")
    out[valid_idx] = proba
    return out.reshape(Y, X)


In [ ]:

# Make per-model maps (create only the ones you need)
map_lr  = predict_map_proba(models["lr"])
map_rf  = predict_map_proba(models["rf"])
map_svm = predict_map_proba(models["svm_rbf"])
map_mlp = predict_map_proba(models["mlp"])


In [ ]:

# Save each as a GeoTIFF using the reference raster metadata, so the output
# lines up spatially with the input rasters (same CRS, transform, extent)
with rasterio.open(REF_RASTER_PATH) as src_ref:
    profile = src_ref.profile
profile.update(dtype="float32", count=1, compress="lzw", nodata=np.nan)


In [ ]:

def write_tif(path, arr, profile=profile):
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)


In [ ]:

# Example filenames
OUT_LR  = "lr_apr.tif"
OUT_RF  = "rf_apr.tif"
OUT_SVM = "svm_apr.tif"
OUT_MLP = "mlp_apr.tif"

write_tif(OUT_LR,  map_lr)
write_tif(OUT_RF,  map_rf)
write_tif(OUT_SVM,  map_svm)
write_tif(OUT_MLP, map_mlp)


## Plot Predicted Maps and Overlay Points

Now we can look at what each model actually predicted across the study area, with the real presence and pseudo-absence points overlaid for a visual sanity check. A well-behaved model should show high suitability concentrated around the presence points and lower suitability elsewhere, without wild, patchy artefacts. Differences between the four maps here are informative in their own right: a smoother, more conservative surface (typically logistic regression) versus a sharper, more localised one (typically random forest or the neural network) reflects the different ways each algorithm generalises beyond the training points.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use any aligned layer to get plotting extent and target CRS
da0 = layers[layer_names[0]]                 # an xarray DataArray (y, x)
target_crs = da0.rio.crs                     # CRS of the rasters
extent = [da0.x.min(), da0.x.max(), da0.y.min(), da0.y.max()]

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colors as mcolors  # optional for norms

# assume: da0, target_crs, extent already defined (from your raster/layers)

def plot_prob_map(
    arr,
    title="April",
    cmap="viridis",        # try: "magma", "plasma", "inferno", "cividis", "YlGn", "YlOrRd"
    vmin=0.0, vmax=1.0,    # clamp color range (great for probabilities)
    pres_color="tab:red",
    abs_color="tab:pink",
    max_absences=300,
    seed=42,
    norm=None              # e.g. mcolors.PowerNorm(gamma=0.5) to emphasise highs
):
    # Reproject points to the raster CRS
    pres_plot = presences.to_crs(target_crs)
    abs_plot  = absences.to_crs(target_crs)
    if len(abs_plot) > max_absences:
        abs_plot = abs_plot.sample(max_absences, random_state=seed)

# Prepare image (mask NaNs)
    img = np.ma.masked_invalid(arr.astype("float32"))

    # Make NaNs transparent (so background shows through, if any)
    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad(alpha=0.0)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        img, extent=extent, origin="upper",
        cmap=cmap_obj, vmin=vmin, vmax=vmax, norm=norm
    )

    pres_plot.plot(ax=ax, color=pres_color, markersize=10, alpha=0.9, label="Presence")
    abs_plot.plot(ax=ax,  color=abs_color,  markersize=6,  alpha=0.6, label="Absence (sample)")

    ax.set_title(title)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.legend(loc="lower right")

    cbar = plt.colorbar(im, ax=ax, shrink=0.75)
    cbar.set_label("Suitability / Probability")
    plt.tight_layout(); plt.show()


In [ ]:

# Compare the four models' suitability maps side by side (run one at a time
# in this cell order): notice where they agree confidently and where they diverge
plot_prob_map(map_lr,  "Logistic Regression: April", cmap = "YlOrRd", pres_color="black", abs_color="blue", max_absences=500)
plot_prob_map(map_rf,  "Random Forest: April", cmap = "viridis_r", pres_color="pink", abs_color="red", max_absences=500)
plot_prob_map(map_svm, "SVM (RBF): April")
plot_prob_map(map_mlp, "MLP: April")


## 8) AUC-Weighted Ensemble and Hold-out Evaluation

No single model here is guaranteed to be the "correct" one, each makes different assumptions and each will get different parts of the environmental relationship right. **Ensembling** combines their predictions instead of picking a single winner, on the reasoning that averaging several imperfect but differently-biased models usually produces a more robust prediction than trusting any one of them alone.

We use an **AUC-weighted average**: each model's prediction is weighted by how well it performed on the test set, so a model that ranked presences well gets more say in the final probability than one that performed poorly. This is a simple but sensible scheme when we have no other basis for choosing weights.

Crucially, we then evaluate this ensemble on the **hold-out** set, which none of the models, and none of the weighting decisions, have seen before. This is the only genuinely unbiased estimate of how the ensemble would perform on new, unsurveyed locations, and it is the number you would actually report if this fed into a real conservation assessment.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use any aligned layer to get plotting extent and target CRS
da0 = layers[layer_names[0]]                 # an xarray DataArray (y, x)
target_crs = da0.rio.crs                     # CRS of the rasters
extent = [da0.x.min(), da0.x.max(), da0.y.min(), da0.y.max()]

In [ ]:

import numpy as np

# Collect AUCs in the SAME order as the maps you computed
w_lr  = aucs["lr"]
w_rf  = aucs["rf"]
w_svm = aucs["svm_rbf"]
w_mlp = aucs["mlp"]

# Put them in a vector and normalise to sum to 1 (avoid divide-by-zero with tiny epsilon).
# This turns each model's raw AUC into a relative "vote share": a model with
# a higher test AUC contributes proportionally more to the ensemble average.
w = np.array([w_lr, w_rf, w_svm, w_mlp], dtype="float64")
w = w / (w.sum() + 1e-9)

print(f"Weights (lr, rf, svm, mlp) = {w.round(3).tolist()}")

# Stack the per-model maps and compute a weighted average
# Using np.average is a bit safer/nicer than manual w[0]*... + w[1]*... + ...
stack_maps = np.stack([map_lr, map_rf, map_svm, map_mlp], axis=0).astype("float32")  # shape: (4, Y, X)
ensemble_map = np.average(stack_maps, axis=0, weights=w).astype("float32")  # shape: (Y, X)


In [ ]:

# Reuse the reference raster profile from earlier
with rasterio.open(REF_RASTER_PATH) as src_ref:
    profile = src_ref.profile
profile.update(dtype="float32", count=1, compress="lzw", nodata=np.nan)

OUT_ENSEMBLE = "ensemble_apr.tif"  # keep your existing name if you had one

def write_tif(path, arr, profile=profile, desc="AUC-weighted ensemble probability"):
    """Write a single-band float32 GeoTIFF with nodata set to NaN."""
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)
        try:
            dst.update_tags(1, DESCRIPTION=desc)
        except Exception:
            pass

write_tif(OUT_ENSEMBLE, ensemble_map)
print("Wrote ensemble map:", OUT_ENSEMBLE)


In [ ]:
layers

In [ ]:

# Call with your ensemble array. This is the map we are most interested in:
# each pixel's value is the AUC-weighted average probability of Atlantic
# sturgeon presence in April, and it is the surface a habitat suitability
# assessment would actually be built from.
plot_prob_map(ensemble_map, title="Atlantic Sturgeon in April: The Ensemble", max_absences=300)


In [ ]:
# Hold-out probabilities from each model (class 1 = Presence). All four models
# contribute here, matching the ensemble map computed earlier from map_lr, map_rf,
# map_svm and map_mlp together.
# Logistic Regression
p_lr = models["lr"].predict_proba(X_hold)[:, 1]

# Random Forest
p_rf = models["rf"].predict_proba(X_hold)[:, 1]

# SVM (RBF), requires probability=True when you created it
p_svm = models["svm_rbf"].predict_proba(X_hold)[:, 1]

# MLP
p_mlp = models["mlp"].predict_proba(X_hold)[:, 1]

In [ ]:
# --- AUC-weighted ensemble on the hold-out set ---
# w was already normalised from the test-set AUCs of all four models (lr, rf, svm, mlp),
# and all four are combined here too, so this matches the ensemble_map computed earlier
# rather than silently dropping one model.
p_ens = (w[0] * p_lr) + (w[1] * p_rf) + (w[2] * p_svm) + (w[3] * p_mlp)
auc_hold = roc_auc_score(y_hold, p_ens)
print(f"Ensemble AUC on hold-out: {auc_hold:.3f}")

In [ ]:

## Fallback for SVM if it lacks predict_proba (not ideal, but keeps values on a [0, 1] scale):
# When probability=True was not set, the SVM only exposes a decision function
# (signed distance from the separating hyperplane), which is not a
# probability. Min-max scaling it into [0, 1] lets it still be combined with
# genuine probabilities in the ensemble, though it is not calibrated the same way.
#dec_svm = models["svm_rbf"].decision_function(X_hold)
#p_svm = (dec_svm - dec_svm.min()) / (dec_svm.max() - dec_svm.min() + 1e-9)


In [ ]:

## THE ENSEMBLE
# Threshold at 0.5 for a simple confusion matrix, turning the continuous
# hold-out ensemble probability into a binary presence/absence call
ens_pred = (p_ens >= 0.5).astype(int)

# Confusion matrix and metrics
cm_ens = confusion_matrix(y_hold, ens_pred)   # [[TN, FP], [FN, TP]]
acc_ens, sens_ens, spec_ens, ppv_ens, npv_ens = metrics_from_cm(cm_ens)

# Note: unlike the AUC values computed earlier in the notebook, this AUC is
# computed on the already-thresholded 0/1 predictions (ens_pred) rather than
# on the continuous probabilities (p_ens). It does not carry the same meaning
# as a proper ROC AUC, since thresholded labels give the ROC curve only two
# points to work with. It is included here for reference, but auc_hold above
# (computed on p_ens directly) is the more meaningful score to compare
# against the other models.
auc_ens = roc_auc_score(y_hold, ens_pred)

print("Ensemble: Confusion matrix [[TN, FP],[FN, TP]]:\n", cm_ens, "\n")
plot_cm(cm_ens, class_names=('Absence','Presence'), title='Ensemble: Confusion Matrix', normalize=False)
print(f"Accuracy={acc_ens:.3f} | Sensitivity={sens_ens:.3f} | Specificity={spec_ens:.3f} | "
      f"PPV={ppv_ens:.3f} | NPV={npv_ens:.3f} | AUC={auc_ens:.3f}")


## 10) Explainable AI: Which Variables Actually Drive the Predictions?

A model that predicts well but cannot say why is a hard sell to the people who actually have to act on it: a regulator deciding whether to permit dredging, a manager prioritising which stretch of river to protect, or a colleague who reasonably wants to know whether the model has learned real ecology or just an artefact of how the data was collected. This is the problem Explainable AI (XAI) addresses: a set of techniques for looking inside a fitted model and asking, in a principled way, which inputs it is actually relying on and how.

We will work through this from the simplest method to the most detailed:

1. **Built-in Random Forest importance**, fast and free, but with a known bias.
2. **Permutation importance**, a more trustworthy, model-agnostic check on (1).
3. **Partial Dependence Plots (PDPs)**, which add back what importance scores cannot tell us: *direction*, does presence go up or down as a variable increases?
4. **SHAP (SHapley Additive exPlanations)**, which goes one level further still and explains individual predictions, not just the model as a whole, so we can ask "why did the model think *this specific location* was suitable?"

Each method answers a slightly different question, and together they tell a much more complete, and much more defensible, story than any one of them alone.

In [ ]:
# --- Method 1: Random Forest built-in feature importances ---
# Fast, and comes "for free" from a model we already fitted, but it is impurity-based:
# it is known to inflate the apparent importance of continuous or high-cardinality
# variables (which get more opportunities to produce a good split) relative to variables
# with less spread, regardless of their real relationship with presence. Treat this as a
# quick first look, not the final word, we sanity-check it against a model-agnostic method next.
rf = models["rf"]
rf_importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(rf_importance)

fig, ax = plt.subplots(figsize=(6, 4))
rf_importance.iloc[:10].plot(kind="bar", ax=ax)  # matplotlib backend via pandas, using top 10 (we don't have that many)
ax.set_title("Random Forest: Built-in Feature Importance")
ax.set_xlabel("Variable")
ax.set_ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Method 2: Permutation Importance

The idea behind permutation importance is simple and intuitive: if a predictor genuinely matters, then scrambling its values (breaking its link to the outcome, while leaving every other predictor untouched) should hurt the model's performance. If scrambling a predictor barely changes performance, the model was not relying on it much. Unlike built-in RF importance, this is model-agnostic (it works on any fitted model) and is measured directly against held-out performance rather than training-time impurity reduction, which makes it a fairer, more trustworthy comparison across variables of different types.

In [ ]:
from sklearn.inspection import permutation_importance

# n_repeats shuffles each column 20 times and averages the drop in performance, giving
# both a mean importance and a sense of how noisy that estimate is (importances_std below)
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=20, random_state=SEED, n_jobs=-1, scoring="roc_auc"
)
perm_importance = pd.Series(perm_result.importances_mean, index=features).sort_values(ascending=False)
perm_std = pd.Series(perm_result.importances_std, index=features)

print(perm_importance)

fig, ax = plt.subplots(figsize=(6, 4))
order = perm_importance.index
ax.bar(order, perm_importance[order], yerr=perm_std[order], capsize=4, color="tab:green")
ax.set_title("Permutation Importance (test set, AUC drop)")
ax.set_xlabel("Variable")
ax.set_ylabel("Mean AUC decrease when shuffled")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Compare rank order with the built-in RF importance above: do the two methods agree
# on the top predictors, or does the impurity-based bias we mentioned actually show up here?
comparison = pd.DataFrame({
    "rf_builtin_rank": rf_importance.rank(ascending=False),
    "permutation_rank": perm_importance.rank(ascending=False),
})
print(comparison.sort_values("permutation_rank"))

### Method 3: Partial Dependence Plots (PDPs)

Importance scores, whichever method produces them, only tell us *how much* a variable matters, not *which way* it points. A Partial Dependence Plot fixes that: it takes one predictor, sweeps it across its observed range while averaging over every other predictor, and plots how the model's predicted probability of presence responds. This is where the ecology starts to show through the model: does predicted suitability rise with depth, fall with it, or peak somewhere in the middle? We plot the two variables that came out on top above, `depth` and `sss_apr` (April sea surface salinity).

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# features= takes integer positions into the `features` list (matching how X_train/X_test
# were built as plain numpy arrays, not DataFrames), and feature_names= supplies readable
# axis labels. Passing indices rather than a re-wrapped DataFrame avoids a spurious
# "fitted without feature names" warning, since rf itself was fitted on a plain array.
top_two = perm_importance.index[:2].tolist()
fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    rf, X_test,
    features=[features.index(top_two[0]), features.index(top_two[1])],
    feature_names=features,
    ax=ax,
)
fig.suptitle("Partial Dependence: Predicted Suitability vs. Top Predictors", y=1.03)
plt.tight_layout()
plt.show()

### Method 4: SHAP (SHapley Additive exPlanations)

The methods so far describe the model's overall behaviour. SHAP lets us go one step further and explain a *single prediction*: for one specific point on the map, how much did each predictor push the probability of sturgeon presence up or down, relative to the average prediction? It is built on Shapley values, a concept from cooperative game theory for fairly splitting credit among contributors, applied here to "how much credit does each predictor deserve for this particular prediction." We use `shap.TreeExplainer`, an efficient exact implementation for tree-based models like Random Forest, to look at both the global pattern (a beeswarm summary across many points) and one individual, real presence location in detail.

*(If `shap` is not already installed in your environment, run `pip install shap` once.)*

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)
shap_values = explainer(X_test)

# For a binary classifier, TreeExplainer returns SHAP values per class; we want the
# "Presence" class (index 1), and we attach readable feature names for the plots below
shap_values_presence = shap_values[..., 1]
shap_values_presence.feature_names = features

# Global view: every test point plotted as a dot per feature, colour shows whether that
# point's value of the feature was high (red) or low (blue), position shows whether it
# pushed the prediction toward presence (right) or absence (left). This is the same
# story as permutation importance and the PDPs above, but in one plot: which variables
# matter (dots spread wide) and which direction they push (colour vs. position)
shap.plots.beeswarm(shap_values_presence, show=False)
plt.title("SHAP Summary: Global Feature Impact on Predicted Presence")
plt.tight_layout()
plt.show()

**Explaining one real prediction.** Global summaries are useful, but the real payoff of SHAP is being able to answer "why did the model predict *this*?" for one specific location, the kind of question a manager will actually ask about a specific site under review. Below we pick a real presence point from the test set and break down exactly how each variable pushed the model's prediction away from the average (the model's baseline expectation before seeing any of this point's predictor values) toward its final probability.

In [ ]:
# Pick a real presence point (Presence == 1) from the test set to explain in detail
presence_idx_in_test = np.where(y_test == 1)[0]
example_idx = presence_idx_in_test[0]

predicted_prob = rf.predict_proba(X_test[[example_idx]])[0, 1]
print(f"Explaining test point {example_idx} (true label: Presence), "
      f"model's predicted probability: {predicted_prob:.3f}")

shap.plots.waterfall(shap_values_presence[example_idx], show=False)
plt.title(f"SHAP Waterfall: Individual Prediction Breakdown (point {example_idx})")
plt.tight_layout()
plt.show()

### Bringing It Together

Notice how the four methods reinforce, rather than contradict, each other: the variables that come out on top in built-in RF importance and permutation importance are the same ones whose partial dependence curves show the clearest trends, and the same ones that dominate the SHAP beeswarm and waterfall plots. That convergence across genuinely different mathematical approaches is itself evidence that the model has learned something real about sturgeon habitat, rather than latching onto noise. When methods disagree sharply instead, that is usually a signal worth investigating (correlated predictors, a small sample, or a model overfitting to a handful of points) rather than something to paper over by picking whichever method tells the nicer story.

## 11) Wrap-Up and Next Steps

- You built a full **geospatial SDM** workflow for April presences:
  - **Buffer-based pseudo-absences**, raster **extraction** to points, a clean modelling table `df_w_var`.
  - **Train/test/hold-out** regimen across four models (logistic regression, random forest, SVM, MLP).
  - An **AUC-weighted ensemble**, thresholded and mapped as **GeoTIFF** predictions.
  - Clear **maps** and a full **Explainable AI** pass: built-in and permutation importance, partial dependence plots, and SHAP (global and per-prediction).

**Next steps / extensions:**
- Try **other algorithms** (gradient boosting, MaxEnt) and compare AUCs against what you have here.
- Add **cross-validation** and **hyperparameter tuning** (for example `GridSearchCV`) rather than relying on a single train/test split.
- Integrate **spatial cross-validation** to control for spatial autocorrelation between nearby presence and absence points, which a random split can understate.
- Move from a single month (April) to **multi-month** or seasonal stacks and ensembles.
- Calibrate the probability threshold properly (for example maximising TSS or F1) rather than defaulting to 0.5.
- Export the maps and build a proper **cartographic layout** suitable for a report.
- Extend the SHAP analysis across the *whole raster stack* (not just the test points) to map, pixel by pixel, which variable is driving predicted suitability everywhere, not only at your sampled locations.